# Lekcia 01 - Úvod do AI agentov

Vitajte v prvej lekcii kurzu **AI agenti pre začiatočníkov**!

**AI agent** je program, ktorý používa veľký jazykový model (LLM) ako svoje rozumové jadro a môže vykonávať *akcie* v reálnom svete — volať API, dotazovať databázy alebo spúšťať kód — aby dosiahol cieľ v mene používateľa.

V tomto notebooku si vybudujete svojho prvého agenta: **Cestovného agenta**, ktorý odporúča dovolenkové destinácie. Počas cesty sa naučíte, ako:

1. Pripojiť sa k službe Azure AI Foundry Agent Service pomocou **Microsoft Agent Framework**.
2. Dodať agentovi **nástroj** — obyčajnú Python funkciu, ktorú môže volať.
3. Spustiť agenta a skontrolovať jeho odpoveď.
4. Streamovať odpoveď agenta token po tokene.


## Nastavenie

Pred spustením tohto notebooku sa uistite, že máte:

1. **Projekt Azure AI Foundry** s nasadeným chatovacím modelom (napr. `gpt-4o-mini`).
2. **Prihlásený cez Azure CLI** — spustite v termináli príkaz `az login`.
3. **Nastavené potrebné premenné prostredia:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint vášho projektu Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — názov vášho nasadeného modelu.

Nižšie uvedená bunka nainštaluje potrebné Python balíčky.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Vytvorenie vášho prvého agenta

Agent potrebuje dve veci:

- **Pokyny**, ktoré mu povedia, *kto je* a *ako sa má správať* (systémový prompt).
- **Nástroje** — Python funkcie označené dekorátorom `@tool`, ktoré môže agent volať na získavanie informácií alebo vykonávanie akcií.

Nižšie definujeme jednoduchý nástroj, ktorý vracia zoznam populárnych dovolenkových destinácií. Agent použije tento nástroj, keď používateľ požiada o odporúčania na cestovanie.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Pre interaktívnejší zážitok môžete **streamovať** odpoveď agenta. Namiesto čakania na celú odpoveď agent postupne poskytuje textové časti, ako sú generované. To je obzvlášť užitočné v chatovacích rozhraniach, kde chcete zobrazovať výstup v reálnom čase.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Zhrnutie

V tejto lekcii ste sa naučili, ako:

- **Vytvoriť poskytovateľa**, ktorý sa pripája k Azure AI Foundry Agent Service prostredníctvom `AzureAIProjectAgentProvider`.
- **Definovať nástroj** pomocou dekorátora `@tool`, aby agent mohol volať vaše Python funkcie.
- **Spustiť agenta** s používateľskou správou a vytlačiť jeho odpoveď.
- **Streamovať odpovede** pre výstup v reálnom čase.

V ďalšej lekcii sa podrobnejšie pozrieme na agentné rámce a naučíme sa, ako dať agentom výkonnejšie nástroje a schopnosti viacstupňového uvažovania.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
